# Legal Agent — Jupyter walkthrough

© Intergrax — internal material.

This notebook demonstrates **practical ways to run** `LegalAgent`: from a simple question to contract analysis with an attachment and RAG. Results depend on a local **Ollama** instance (chat + embedding models per project configuration).

**Contents**
1. Environment setup and shared helpers
2. Scenario A — legal question without a file (dynamic pipeline)
3. Scenario B — contract text as an attachment + vector RAG
4. Scenario C — Tier-2 tool decision enabled (`use_legal_tool_decision`) + trace excerpt


## Prerequisites

- **Ollama** running (`http://127.0.0.1:11434` or `OLLAMA_HOST`).
- Python environment with the **intergrax** package installed (e.g. `pip install -e .` from the repository root), or the **project root on `sys.path`** (see the code cell below).
- Recommended: `pandas` for trace tables (already a project dependency).

> If `import intergrax` fails, start Jupyter from the **repository root** (where `pyproject.toml` lives) or adjust the path logic in the first code cell.


In [ ]:
from __future__ import annotations

import asyncio
import os
import sys
from pathlib import Path

# Ensure repository root + agents/ on sys.path when not using pip install -e .
_here = Path.cwd()
for _p in (_here, *_here.parents):
    if (_p / "pyproject.toml").is_file() and (_p / "intergrax").is_dir():
        _root = str(_p.resolve())
        if _root not in sys.path:
            sys.path.insert(0, _root)
        _agents = str((_p / "agents").resolve())
        if (_p / "agents").is_dir() and _agents not in sys.path:
            sys.path.insert(0, _agents)
        break

from intergrax.agents.agent_engine import AgentEngine
from legal.legal_agent import LegalAgent
from legal.config.legal_agent_config import LegalAgentConfig
from intergrax.llm.messages import AttachmentRef
from intergrax.llm_adapters.contracts.llm_provider import LLMProvider
from intergrax.llm_adapters.llm_provider_registry import LLMAdapterRegistry
from intergrax.rag.embedding.bootstrap.default_embedding_engine import create_default_embedding_pipeline
from intergrax.rag.embedding.embedding_manager import EmbeddingManager
from intergrax.rag.vectorstore.bootstrap.vectorstore_bootstrap import create_default_vectorstore_manager
from intergrax.runtime.nexus.responses.response_schema import RuntimeRequest
from intergrax.runtime.nexus.session.in_memory_session_storage import InMemorySessionStorage
from intergrax.runtime.nexus.session.session_manager import SessionManager

import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(obj, **_):
        print(obj)

print("Python:", sys.version.split()[0], "| cwd:", _here)
print("OLLAMA_HOST:", os.environ.get("OLLAMA_HOST", "(default 127.0.0.1:11434)"))


## Architecture (short)

- **`LegalAgent`** builds `RuntimeContext`: LLM adapter, optional RAG (embedding + vectorstore), websearch, tools, governance.
- **Pipeline**: by default `LegalDynamicPipeline` — legal stages are chosen dynamically; with `enable_sequential_legal_pipeline=True` you get a fixed-order `LegalAnalysisPipeline`.
- **`use_legal_tool_decision=True`** enables the Tier-2 tool decision, then the Nexus bridge (`RagStep` / `WebsearchStep` / `ToolsStep`) before legal stages.
- **Governance**: `organization_allow_*` and optionally `legal_tool_plan_governance` can clamp layers before the bridge.

The contract demo below sets **`use_llm_legal_route_planner=False`** on purpose so more stages run deterministically; scenario A keeps the default LLM route planner.


In [ ]:
def build_session_manager() -> SessionManager:
    return SessionManager(storage=InMemorySessionStorage())


def build_ollama_legal_config(
    *,
    tenant_id: str,
    enable_rag: bool = False,
    use_legal_tool_decision: bool = False,
    use_llm_legal_route_planner: bool = True,
    use_legal_run_evaluator: bool = True,
    production_mode: bool = False,
) -> tuple[LegalAgentConfig, LegalAgent]:
    llm_adapter = LLMAdapterRegistry.create(LLMProvider.OLLAMA)
    embedding_manager = None
    vectorstore_manager = None
    if enable_rag:
        embedding_manager = EmbeddingManager(pipeline=create_default_embedding_pipeline(provider_id="ollama"))
        vectorstore_manager = create_default_vectorstore_manager(tenant_id=tenant_id)

    cfg = LegalAgentConfig(
        session_manager=build_session_manager(),
        llm_adapter=llm_adapter,
        production_mode=production_mode,
        enable_rag=enable_rag,
        embedding_manager=embedding_manager,
        vectorstore_manager=vectorstore_manager,
        use_legal_tool_decision=use_legal_tool_decision,
        use_llm_legal_route_planner=use_llm_legal_route_planner,
        use_legal_run_evaluator=use_legal_run_evaluator,
    )
    return cfg, LegalAgent(config=cfg)


def run_legal(agent: LegalAgent, request: RuntimeRequest):
    """Synchronous wrapper (handy in Jupyter)."""
    return asyncio.run(AgentEngine.run_agent(agent, request))


def show_answer(result, *, max_chars: int = 4000) -> None:
    text = (result.answer or "").strip()
    if len(text) > max_chars:
        text = text[:max_chars] + "\n\n… [truncated]"
    print(text)
    print("\n--- route ---")
    r = result.route
    print("strategy:", getattr(r, "strategy", None))
    print("used_rag:", getattr(r, "used_rag", None), "| used_websearch:", getattr(r, "used_websearch", None), "| used_tools:", getattr(r, "used_tools", None))


def trace_dataframe(result) -> pd.DataFrame:
    rows = []
    for e in result.trace_events or []:
        lvl = e.level.value if hasattr(e.level, "value") else e.level
        comp = e.component.value if hasattr(e.component, "value") else e.component
        rows.append({"seq": e.seq, "component": comp, "step": e.step, "level": lvl, "message": (e.message or "")[:200]})
    return pd.DataFrame(rows)


## Scenario A — general legal question (no attachment)

The agent answers from model knowledge and the **internal legal pipeline** (no input file). The default **LLM route planner** is on—which stages run (extract, risk, decision, …) depends on the model and the question.

**What to watch:** answer length, `route.strategy`, and the trace step list.


In [ ]:
tenant_a = "notebook-legal-demo-a"
cfg_a, agent_a = build_ollama_legal_config(tenant_id=tenant_a, enable_rag=False)

req_a = RuntimeRequest(
    agent_id="legal-notebook-a",
    user_id="demo-user",
    session_id="demo-session-a",
    message=(
        "Briefly: what are typical elements of a confidentiality clause in an NDA "
        "(no analysis of a specific document)?"
    ),
    attachments=[],
    tenant_id=tenant_a,
    workspace_id="ws-demo",
)

result_a = run_legal(agent_a, req_a)
show_answer(result_a)
trace_dataframe(result_a).head(25)


## Scenario B — contract file + RAG

We create a **temporary text file** with clauses, attach it via `AttachmentRef` (`file://` URI), and enable **RAG** (Ollama embedding + default vectorstore). For a repeatable demo we turn off the LLM route planner (`use_llm_legal_route_planner=False`) so the pipeline runs the full stage set (except stages that self-skip).

**What to watch:** `route.used_rag == True`, extract/analysis-related trace steps, and references to the file content in the answer.


In [ ]:
from tempfile import TemporaryDirectory

tenant_b = "notebook-legal-demo-b"
cfg_b, agent_b = build_ollama_legal_config(
    tenant_id=tenant_b,
    enable_rag=True,
    use_llm_legal_route_planner=False,
    use_legal_run_evaluator=False,
)

with TemporaryDirectory() as tmp:
    contract_path = Path(tmp) / "sample_contract.txt"
    contract_path.write_text(
        """
        Non-disclosure agreement (demo excerpt)

        1. The parties shall keep Confidential Information secret for 3 years from disclosure.
        2. The supplier shall not be liable for indirect damages or lost profits.
        3. The client agrees to pay within 14 days of the invoice date.
        """.strip(),
        encoding="utf-8",
    )
    attachment = AttachmentRef(
        id="demo-contract",
        type="txt",
        uri=contract_path.resolve().as_uri(),
    )
    req_b = RuntimeRequest(
        agent_id="legal-notebook-b",
        user_id="demo-user",
        session_id="demo-session-b",
        message="Analyze the attached contract: list the main risks and payment terms.",
        attachments=[attachment],
        tenant_id=tenant_b,
        workspace_id="ws-demo",
    )
    result_b = run_legal(agent_b, req_b)

show_answer(result_b)
trace_dataframe(result_b)


## Scenario C — Tier-2 tool decision + Nexus bridge (optional)

With **`use_legal_tool_decision=True`**, an LLM tool decision runs before legal stages, then—depending on the plan—Nexus steps: RAG / websearch / tools.

This section does **not** register tools or websearch; it only shows how to flip the flag and what appears in trace (e.g. `LegalToolDecision`). In production you wire `ToolsAgent`, `WebSearchExecutor`, etc. (see e2e tests under `applications/legal_agent/tests/`).


In [ ]:
tenant_c = "notebook-legal-demo-c"
cfg_c, agent_c = build_ollama_legal_config(
    tenant_id=tenant_c,
    enable_rag=True,
    use_legal_tool_decision=True,
    use_llm_legal_route_planner=True,
    use_legal_run_evaluator=True,
)

req_c = RuntimeRequest(
    agent_id="legal-notebook-c",
    user_id="demo-user",
    session_id="demo-session-c",
    message="For a simple question about the definition of GDPR good practice, do you need full RAG? Answer briefly.",
    attachments=[],
    tenant_id=tenant_c,
    workspace_id="ws-demo",
)

result_c = run_legal(agent_c, req_c)
show_answer(result_c)
tdf = trace_dataframe(result_c)
if len(tdf) == 0:
    display(tdf)
else:
    mask = tdf["step"].str.contains("Legal|Tool|rag|Rag", case=False, na=False)
    display(tdf[mask] if mask.any() else tdf.head(20))


## Governance (organization)

In `LegalAgentConfig` you can set for example:

- `organization_allow_rag` / `organization_allow_websearch` / `organization_allow_tools` — static plan clamp before the bridge (trace events `LegalToolPlanGovernance`).
- `legal_tool_plan_governance` — port for dynamic plan adjustment (implementations in `legal_tool_plan_governance_impl.py`).
- `organization_compliance_policy` — policy text for the compliance step.

Experiment: duplicate `build_ollama_legal_config(...)`, set `organization_allow_rag=False`, and force `use_rag` in a test (e.g. patch as in `test_legal_tool_plan_governance.py`)—in the notebook you can compare trace with `True` vs `False`.

---

**Next steps:** run e2e tests under `tests/` and `applications/legal_agent/tests/` for full websearch/tools scenarios.
